<a href="https://colab.research.google.com/github/AndrewDem04/web-services-fundamentals/blob/main/movie_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PySpark Install**

In [ ]:
!pip install pyspark
!apt install openjdk-8-jdk

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-common at-spi2-core fonts-dejavu-core fonts-dejavu-extra
  fonts-dejavu-mono gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0t64 libatspi2.0-0t64
  libgail-common libgail18t64 libgtk2.0-0t64 libgtk2.0-bin libgtk2.0-common
  librsvg2-common libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-8-jdk-headless openjdk-8-jre openjdk-8-jre-headless
  session-migration x11-utils
Suggested packages:
  gvfs libxt-doc openjdk-8-demo openjdk-8-source visualvm libnss-mdns
  fonts-nanum fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei fonts-indic mesa-utils
Recommended packages:
  luit
The following NEW packages will be installed:
  at-spi2-common at-spi2-core fonts-dejavu-core fonts-dejavu-extra
  fonts-dejavu-mono gsettings-desktop-schemas libatk-br

**SparkSession Initialize**

In [ ]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Basic DataFrame Lab") \
    .getOrCreate()

print("SparkSession initiated")

SparkSession initiated


**Load the  movies.json File, have Spark infer the data types.**

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded.keys()))
df = spark.read.json(filename, multiLine=True)
df.show(10)

Saving movies.json to movies.json
+-----------+------------+------------------+--------------------+
|      genre|release_year|             title|               views|
+-----------+------------+------------------+--------------------+
|     Sci-Fi|        2021| The Quantum Heist|[1050, 980, 1200,...|
|      Drama|        2019|  Midnight Whisper|[800, 850, 900, 1...|
|     Sci-Fi|        2023|    Galactic Drift|[2500, 2400, 2200...|
|     Comedy|        2020|Chuckles and Tears|[600, 620, 590, 6...|
|     Action|        2018| The Crimson Blade|[1500, 1450, 1600...|
|  Animation|        2022|      Paws of Fury|[3000, 3100, 2900...|
|   Thriller|        2017|   City of Shadows|[900, 880, 910, 9...|
|Documentary|        2021|     Ocean's Depth|[400, 420, 450, 4...|
|  Adventure|        2015|   The Lost Empire|[1100, 1150, 1120...|
|    Romance|        2016|     Love in Paris|[1300, 1500, 1400...|
+-----------+------------+------------------+--------------------+
only showing top 10 rows


In [ ]:
df.columns
df.printSchema()
df.head(5)
df.describe().show()

root
 |-- genre: string (nullable = true)
 |-- release_year: long (nullable = true)
 |-- title: string (nullable = true)
 |-- views: array (nullable = true)
 |    |-- element: long (containsNull = true)

+-------+--------+------------------+------------------+
|summary|   genre|      release_year|             title|
+-------+--------+------------------+------------------+
|  count|      20|                20|                20|
|   mean|    NULL|           2019.85|              NULL|
| stddev|    NULL|2.9607075967890135|              NULL|
|    min|  Action|              2014|Chuckles and Tears|
|    max|Thriller|              2025| The Quantum Heist|
+-------+--------+------------------+------------------+



In [ ]:
import pyspark.sql.functions as F

movies_total = df.withColumn(
    "total_views",
    F.aggregate("views", F.lit(0).cast("long"), lambda acc, x: acc + x)
)
movies_total.show()

+-----------+------------+------------------+--------------------+-----------+
|      genre|release_year|             title|               views|total_views|
+-----------+------------+------------------+--------------------+-----------+
|     Sci-Fi|        2021| The Quantum Heist|[1050, 980, 1200,...|      13370|
|      Drama|        2019|  Midnight Whisper|[800, 850, 900, 1...|      11590|
|     Sci-Fi|        2023|    Galactic Drift|[2500, 2400, 2200...|      28820|
|     Comedy|        2020|Chuckles and Tears|[600, 620, 590, 6...|       8220|
|     Action|        2018| The Crimson Blade|[1500, 1450, 1600...|      18350|
|  Animation|        2022|      Paws of Fury|[3000, 3100, 2900...|      37500|
|   Thriller|        2017|   City of Shadows|[900, 880, 910, 9...|      11230|
|Documentary|        2021|     Ocean's Depth|[400, 420, 450, 4...|       5340|
|  Adventure|        2015|   The Lost Empire|[1100, 1150, 1120...|      13590|
|    Romance|        2016|     Love in Paris|[1300, 